# Demo: RAG Evaluation & Use Cases
### Module 5, Topic 7 — RAG from Scratch

**What you'll see in this notebook:**
1. Build a small labelled test set — questions with a known "correct" chunk
2. Measure retrieval quality: did the correct chunk show up in the top-k?
3. Run the full pipeline and manually check answers for faithfulness against their cited source
4. Deliberately run a tricky question and see how the pipeline handles a genuine edge case

This is the last notebook in the module — everything here evaluates the exact pipeline built across Topics 1–6.


## Step 0 — Rebuild the Full Pipeline From Topic 6

Same knowledge base, same retriever, same improved prompt with citations, same relevance threshold.

In [ ]:
!pip install anthropic voyageai --quiet

In [ ]:
import os
import anthropic
import voyageai
import numpy as np

claude = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

CHAT_MODEL = "claude-3-5-sonnet-20241022"
EMBED_MODEL = "voyage-4"
RELEVANCE_THRESHOLD = 0.4

chunks = [
    "Naija One Bank — Flexi Save Account Policy (Effective 2026)\n\nThe Flexi Save account is Naija One Bank's flagship savings product for individual customers.",
    "It is designed for customers who want easy access to their funds while still earning competitive interest.",
    "The account has no monthly maintenance fee as long as the minimum balance is maintained.",
    "Interest is calculated daily and credited monthly at a rate of 4.2% per annum.",
    "The minimum opening balance required to activate the account is NGN 5,000.",
    "To continue earning interest, customers must maintain a minimum balance of NGN 1,000 at all times.",
    "Customers are permitted 3 free withdrawals per month. A fee of NGN 500 applies to each withdrawal beyond this limit.",
    "Withdrawals can be made via the mobile app, at any branch, or through an ATM using the Flexi Save debit card.",
    "Accounts that fall below the minimum balance for more than 60 consecutive days will be automatically converted to a Basic Save account, which does not earn interest.",
    "Customers can reactivate Flexi Save status by restoring the minimum balance.",
]

chunk_embeddings = vo.embed(chunks, model=EMBED_MODEL, input_type="document").embeddings
knowledge_base = list(zip(chunks, chunk_embeddings))

def cosine_similarity(vec_a, vec_b):
    vec_a = np.array(vec_a)
    vec_b = np.array(vec_b)
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def retrieve(query, knowledge_base, k=3):
    query_embedding = vo.embed([query], model=EMBED_MODEL, input_type="query").embeddings[0]
    scores = []
    for chunk_text, chunk_vec in knowledge_base:
        score = cosine_similarity(query_embedding, chunk_vec)
        scores.append((chunk_text, score))
    ranked = sorted(scores, key=lambda pair: pair[1], reverse=True)
    return ranked[:k]

def build_prompt(question, retrieved_chunks):
    numbered_context = "\n\n".join(
        f"[Source {i+1}]\n{chunk_text}"
        for i, (chunk_text, score) in enumerate(retrieved_chunks)
    )
    return f"""Use only the information in the SOURCES below to answer the QUESTION.
If the SOURCES do not contain enough information to answer, say so clearly instead of guessing.
After your answer, on a new line, state which source number(s) you used, like this: Source: 2

SOURCES:
{numbered_context}

QUESTION:
{question}
"""

def is_relevant_enough(retrieved_chunks, threshold=RELEVANCE_THRESHOLD):
    if not retrieved_chunks:
        return False
    return retrieved_chunks[0][1] >= threshold

def answer_question(question, knowledge_base):
    retrieved_chunks = retrieve(question, knowledge_base, k=3)
    if not is_relevant_enough(retrieved_chunks):
        return "I don't have information about that in the Flexi Save policy document. Please contact Naija One Bank support directly.", retrieved_chunks
    prompt = build_prompt(question, retrieved_chunks)
    response = claude.messages.create(
        model=CHAT_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text, retrieved_chunks

print("Full pipeline rebuilt.")

## Step 1 — Build a Small Labelled Test Set

For each question, we note the index (0-based) of the chunk that *should* answer it. This is what makes evaluation possible — without a known correct answer, there's nothing to check against.

In [ ]:
test_set = [
    {"question": "How much can I withdraw for free each month before I'm charged?", "expected_chunk_index": 6},
    {"question": "What interest rate do I earn on my savings?", "expected_chunk_index": 3},
    {"question": "How much money do I need to open the account?", "expected_chunk_index": 4},
    {"question": "What happens if my balance stays too low for two months?", "expected_chunk_index": 8},
]

for item in test_set:
    print(f"Q: {item['question']}")
    print(f"   Expected chunk: [{item['expected_chunk_index']}] {chunks[item['expected_chunk_index']][:60]}...\n")

## Step 2 — Evaluate Retrieval Quality

For each test question, run the retriever and check whether the expected chunk appears anywhere in the top-3 results.

In [ ]:
def evaluate_retrieval(test_set, knowledge_base, k=3):
    results = []
    for item in test_set:
        retrieved = retrieve(item["question"], knowledge_base, k=k)
        retrieved_texts = [text for text, score in retrieved]
        expected_text = chunks[item["expected_chunk_index"]]
        found = expected_text in retrieved_texts
        results.append({"question": item["question"], "found": found})
    return results

retrieval_results = evaluate_retrieval(test_set, knowledge_base)

hits = sum(1 for r in retrieval_results if r["found"])
print(f"Retrieval score: {hits}/{len(retrieval_results)}\n")
for r in retrieval_results:
    status = "FOUND" if r["found"] else "MISSED"
    print(f"[{status}] {r['question']}")

## Step 3 — Run the Full Pipeline on Each Question

Now that retrieval quality is checked, let's see what the whole system actually answers.

In [ ]:
for item in test_set:
    answer, retrieved = answer_question(item["question"], knowledge_base)
    print(f"Q: {item['question']}")
    print(f"A: {answer}")
    print()

## Step 4 — Check Faithfulness by Hand

Pick one answer and read it directly against the source chunk it cited. This is the manual faithfulness check from the slides — is every specific number and fact in the answer actually present in that chunk?

In [ ]:
check_question = test_set[0]["question"]
answer, retrieved = answer_question(check_question, knowledge_base)

print("ANSWER:")
print(answer)
print()
print("RETRIEVED SOURCES (what the answer was built from):")
for i, (chunk_text, score) in enumerate(retrieved):
    print(f"[Source {i+1}] (score: {score:.4f})")
    print(chunk_text)
    print()

## Step 5 — Read Them Side by Side

Compare the answer's specific claims — the number of free withdrawals, the fee amount — against the source text printed above. Every specific number in the answer should trace back to text that's actually there. If it doesn't, that's a faithfulness failure, regardless of how confident the answer sounds.

## Step 6 — Deliberately Try a Tricky Question

This question sits in a grey area: it's related to the document's topic (accounts, individual customers) but asks something the document never directly addresses.

In [ ]:
tricky_question = "Can I open a Flexi Save account for my small business?"

answer, retrieved = answer_question(tricky_question, knowledge_base)

print("ANSWER:")
print(answer)
print()
print("RETRIEVED SOURCES:")
for i, (chunk_text, score) in enumerate(retrieved):
    print(f"[Source {i+1}] (score: {score:.4f})")
    print(chunk_text)
    print()

## Step 7 — Diagnose What Happened

This is exactly the kind of case flagged in the slides as genuinely hard to predict in advance:

- The document mentions "individual customers" once, near the top — enough word/topic overlap that this chunk may score above the relevance threshold even though it doesn't actually answer the business-account question
- If the top-scoring chunk passes the threshold, the model receives it as context. A well-instructed model should notice the chunk doesn't actually confirm or deny business accounts and say so — but this is exactly the kind of borderline case where prompt wording (Topic 6) and threshold tuning (Topic 5) both matter, and where a single spot-check isn't enough to be confident

**Using the diagnostic table from the slides:** if this answer confidently claims something the source doesn't support, that's a generation/faithfulness failure — the fix is sharpening the prompt further. If retrieval never surfaces anything reasonably related at all, that's a retrieval limitation — the fix would be expanding the knowledge base to actually cover business accounts.

## Step 8 — What a Real Evaluation Process Looks Like

What we did by hand in this notebook — a handful of questions, checked one at a time — is the manual starting point. A real evaluation process would:

- Grow the test set to dozens or hundreds of questions, covering edge cases like Step 6-7 deliberately
- Re-run it every time the chunking strategy, prompt, or retriever changes
- Track the retrieval score and a faithfulness score over time, not just check them once

The mechanics don't change — only the scale.

## Module Complete

Across this module, a full RAG system was built entirely from first principles:

- **Topic 1** — why RAG is needed at all
- **Topic 2** — loading and chunking documents
- **Topic 3** — a custom retriever, built from scratch
- **Topic 4** — the same retriever, rebuilt with LangChain
- **Topic 5** — connecting the retriever to an LLM call
- **Topic 6** — sharpening the prompt for faithfulness and citations
- **Topic 7** — evaluating whether the whole thing actually works

Every piece of this — the Naija One Bank knowledge base, the retriever, the prompt, the evaluation approach — generalises directly to real RAG systems across banking, legal, healthcare, government, and e-commerce applications.